In [ ]:
import os, glob, json
import unicodedata
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Paths
DATA_PATH = '/kaggle/input/diacritics/dataset'
TRAIN_FILE = os.path.join(DATA_PATH, 'train.txt')
VAL_FILE = os.path.join(DATA_PATH, 'val.txt')

# Base hyperparameters
MAXLEN = 500
EMBED_DIM = 25
LSTM_UNITS = 256
FF_UNITS = 512
DROPOUT = 0.5
BATCH_SIZE = 64
EPOCHS = 40
PLACEHOLDER = '<NONAR>'
PAD_TOKEN = '<PAD>'
SOS_TOKEN = '<SOS>'
EOS_TOKEN = '<EOS>'
UNK_TOKEN = '<UNK>'
SPACE_TOKEN = '<SPACE>'

FEATURE_EXPERIMENTS = [
    {
        'name': 'Baseline: Character Embeddings Only',
        'use_ngram': False,
        'use_char_freq': False,
        'use_word_boundary': False,
    },
    {
        'name': 'With N-gram Features',
        'use_ngram': True,
        'use_char_freq': False,
        'use_word_boundary': False,
    },
    {
        'name': 'With Character Frequency',
        'use_ngram': False,
        'use_char_freq': True,
        'use_word_boundary': False,
    },
    {
        'name': 'With Word Boundaries',
        'use_ngram': False,
        'use_char_freq': False,
        'use_word_boundary': True,
    },
    {
        'name': 'All Features Combined',
        'use_ngram': True,
        'use_char_freq': True,
        'use_word_boundary': True,
    },
]

print(f"\n Running {len(FEATURE_EXPERIMENTS)} feature experiments")
print(f" {EPOCHS} epochs per experiment for quick testing\n")


# ============================================================================
#  Preprocessing 
# ============================================================================

def is_combining(ch):
    return unicodedata.category(ch) == 'Mn'

def split_char_diacritic_pairs(sentence):
    pairs = []
    base = None
    diacs = ''
    for ch in sentence:
        if is_combining(ch):
            if base is None:
                base = '<UNK_BASE>'
            diacs += ch
        else:
            if base is not None:
                pairs.append((base, diacs))
            base = ch
            diacs = ''
    if base is not None:
        pairs.append((base, diacs))
    return pairs

def is_arabic_letter(ch):
    if not isinstance(ch, str) or len(ch) != 1:
        return False
    code = ord(ch)
    return (
        (0x0600 <= code <= 0x06FF) or
        (0x0750 <= code <= 0x077F) or
        (0x08A0 <= code <= 0x08FF) or
        (0xFB50 <= code <= 0xFDFF) or
        (0xFE70 <= code <= 0xFEFF)
    )

def placeholder_transform_pairs(pairs, placeholder=PLACEHOLDER):
    tokens = []
    labels = []
    for base, d in pairs:
        if isinstance(base, str) and base.startswith('<') and base.endswith('>'):
            tokens.append(base)
            labels.append('')
        elif base.isspace():
            tokens.append(SPACE_TOKEN)
            labels.append('')
        elif is_arabic_letter(base) or base == 'ـ':
            tokens.append(base)
            labels.append(d)
        else:
            tokens.append(placeholder)
            labels.append('')
    return tokens, labels


# ============================================================================
# Load and Prepare Data
# ============================================================================

def load_lines_from_file(path):
    lines = []
    with open(path, 'r', encoding='utf8') as fh:
        for line in fh:
            s = line.strip()
            if s:
                lines.append(s)
    return lines

print('Loading data...')
train_lines = load_lines_from_file(TRAIN_FILE) if os.path.exists(TRAIN_FILE) else []
val_lines = load_lines_from_file(VAL_FILE) if os.path.exists(VAL_FILE) else []
print(f'Train lines: {len(train_lines)}, Val lines: {len(val_lines)}')

train_tokens, train_labels = [], []
for line in train_lines:
    pairs = split_char_diacritic_pairs(line)
    toks, labs = placeholder_transform_pairs(pairs)
    train_tokens.append(toks)
    train_labels.append(labs)

val_tokens, val_labels = [], []
for line in val_lines:
    pairs = split_char_diacritic_pairs(line)
    toks, labs = placeholder_transform_pairs(pairs)
    val_tokens.append(toks)
    val_labels.append(labs)


# ============================================================================
# Build Vocabularies 
# ============================================================================

char_counter = Counter(token for seq in (train_tokens + val_tokens) for token in seq)
special_chars = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN, SPACE_TOKEN, PLACEHOLDER]
single_chars = sorted([c for c in char_counter if len(c) == 1 and c not in special_chars])
multi_chars = sorted([c for c in char_counter if len(c) != 1 and c not in special_chars])

chars = special_chars + single_chars + multi_chars
char2idx = {c:i for i,c in enumerate(chars)}
idx2char = {i:c for c,i in char2idx.items()}

all_diacs = [d for lab in train_labels for d in lab]
diac_counter = Counter(all_diacs)
diac_classes = ['<PAD_LABEL>', '<NONE>']
diac_classes.extend(d for d,_ in diac_counter.items() if d != '')
diac2idx = {d:i for i,d in enumerate(diac_classes)}
idx2diac = {i:d for d,i in diac2idx.items()}

print(f'Vocab size: {len(char2idx)}, Diac labels: {len(diac2idx)}')


# ============================================================================
# Feature Extraction Functions
# ============================================================================

def compute_char_frequencies(tokens_list):
    """TF-IDF style character frequencies"""
    char_counts = {}
    total = 0
    for tokens in tokens_list:
        for tok in tokens:
            char_counts[tok] = char_counts.get(tok, 0) + 1
            total += 1
    return {c: count/total for c, count in char_counts.items()}

def get_ngram_features(tokens, position):
    """Bigram and trigram IDs"""
    prev_tok = tokens[position-1] if position > 0 else '<START>'
    curr_tok = tokens[position]
    next_tok = tokens[position+1] if position < len(tokens)-1 else '<END>'
    
    bigram_id = hash(prev_tok + curr_tok) % 1000
    trigram_id = hash(prev_tok + curr_tok + next_tok) % 1000
    return bigram_id, trigram_id

def get_word_boundary_features(tokens, position):
    """Distance to nearest word boundaries"""
    dist_prev = 0
    for i in range(position - 1, -1, -1):
        if tokens[i] == SPACE_TOKEN or tokens[i] == PLACEHOLDER:
            break
        dist_prev += 1
    
    dist_next = 0
    for i in range(position + 1, len(tokens)):
        if tokens[i] == SPACE_TOKEN or tokens[i] == PLACEHOLDER:
            break
        dist_next += 1
    
    return min(dist_prev, 20), min(dist_next, 20)


# ============================================================================
# Enhanced Dataset with Feature Extraction
# ============================================================================

class FeatureDataset(Dataset):
    def __init__(self, tokens_list, labels_list, char_freqs, feature_config, maxlen=MAXLEN):
        self.tokens_list = tokens_list
        self.labels_list = labels_list
        self.char_freqs = char_freqs
        self.feature_config = feature_config
        self.maxlen = maxlen
    
    def __len__(self):
        return len(self.tokens_list)
    
    def __getitem__(self, idx):
        tokens = self.tokens_list[idx]
        labels = self.labels_list[idx]
        
        # Add SOS/EOS
        seq_mod = [SOS_TOKEN] + tokens + [EOS_TOKEN]
        lab_mod = [''] + labels + ['']
        
        # Feature 1: Character IDs (baseline)
        x_ids = [char2idx.get(t, char2idx[UNK_TOKEN]) for t in seq_mod]
        y_ids = [diac2idx.get(d, diac2idx['<NONE>']) if d == '' else diac2idx.get(d, diac2idx['<NONE>']) for d in lab_mod]
        
        # Feature 2: N-gram features
        ngram_bi = []
        ngram_tri = []
        if self.feature_config['use_ngram']:
            for i in range(len(seq_mod)):
                bi, tri = get_ngram_features(seq_mod, i)
                ngram_bi.append(bi)
                ngram_tri.append(tri)
        else:
            ngram_bi = [0] * len(x_ids)
            ngram_tri = [0] * len(x_ids)
        
        # Feature 3: Character frequency
        char_freq_ids = []
        if self.feature_config['use_char_freq']:
            for tok in seq_mod:
                freq = self.char_freqs.get(tok, 0.0)
                freq_bin = min(int(freq * 10000), 99)
                char_freq_ids.append(freq_bin)
        else:
            char_freq_ids = [0] * len(x_ids)
        
        # Feature 4: Word boundaries
        boundary_prev = []
        boundary_next = []
        if self.feature_config['use_word_boundary']:
            for i in range(len(seq_mod)):
                dp, dn = get_word_boundary_features(seq_mod, i)
                boundary_prev.append(dp)
                boundary_next.append(dn)
        else:
            boundary_prev = [0] * len(x_ids)
            boundary_next = [0] * len(x_ids)
        
        # Truncate
        if len(x_ids) > self.maxlen:
            x_ids = x_ids[:self.maxlen]
            y_ids = y_ids[:self.maxlen]
            ngram_bi = ngram_bi[:self.maxlen]
            ngram_tri = ngram_tri[:self.maxlen]
            char_freq_ids = char_freq_ids[:self.maxlen]
            boundary_prev = boundary_prev[:self.maxlen]
            boundary_next = boundary_next[:self.maxlen]
        
        # Pad
        pad_len = self.maxlen - len(x_ids)
        x_ids += [char2idx[PAD_TOKEN]] * pad_len
        y_ids += [diac2idx['<PAD_LABEL>']] * pad_len
        ngram_bi += [0] * pad_len
        ngram_tri += [0] * pad_len
        char_freq_ids += [0] * pad_len
        boundary_prev += [0] * pad_len
        boundary_next += [0] * pad_len
        
        return (
            torch.tensor(x_ids, dtype=torch.long),
            torch.tensor(ngram_bi, dtype=torch.long),
            torch.tensor(ngram_tri, dtype=torch.long),
            torch.tensor(char_freq_ids, dtype=torch.long),
            torch.tensor(boundary_prev, dtype=torch.long),
            torch.tensor(boundary_next, dtype=torch.long),
            torch.tensor(y_ids, dtype=torch.long),
        )


# ============================================================================
# Enhanced BiLSTM Model
# ============================================================================

class EnhancedBiLSTM(nn.Module):
    def __init__(self, vocab_size, num_labels, feature_config, embed_dim=EMBED_DIM, 
                    lstm_units=LSTM_UNITS, ff_units=FF_UNITS, dropout=DROPOUT):
        super().__init__()
        self.feature_config = feature_config
        
        # Feature 1: Character embeddings
        self.char_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=char2idx[PAD_TOKEN])
        total_dim = embed_dim
        
        # Feature 2: N-gram embeddings
        if feature_config['use_ngram']:
            self.bigram_emb = nn.Embedding(1000, 16, padding_idx=0)
            self.trigram_emb = nn.Embedding(1000, 16, padding_idx=0)
            total_dim += 32
        
        # Feature 3: Character frequency
        if feature_config['use_char_freq']:
            self.freq_emb = nn.Embedding(100, 8, padding_idx=0)
            total_dim += 8
        
        # Feature 4: Word boundaries
        if feature_config['use_word_boundary']:
            self.boundary_prev_emb = nn.Embedding(21, 8, padding_idx=0)
            self.boundary_next_emb = nn.Embedding(21, 8, padding_idx=0)
            total_dim += 16
        
        self.bilstm1 = nn.LSTM(total_dim, lstm_units, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout1 = nn.Dropout(dropout)
        self.bilstm2 = nn.LSTM(2*lstm_units, lstm_units, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout2 = nn.Dropout(dropout)
        
        self.ff1 = nn.Linear(2*lstm_units, ff_units)
        self.ff2 = nn.Linear(ff_units, ff_units)
        self.out = nn.Linear(ff_units, num_labels)
        self.relu = nn.ReLU()
    
    def forward(self, x_ids, ngram_bi, ngram_tri, char_freq, bound_prev, bound_next):
        # Feature 1: Character embeddings
        emb = self.char_embedding(x_ids)
        
        # Feature 2: N-grams
        if self.feature_config['use_ngram']:
            bi_emb = self.bigram_emb(ngram_bi)
            tri_emb = self.trigram_emb(ngram_tri)
            emb = torch.cat([emb, bi_emb, tri_emb], dim=-1)
        
        # Feature 3: Frequency
        if self.feature_config['use_char_freq']:
            freq_emb = self.freq_emb(char_freq)
            emb = torch.cat([emb, freq_emb], dim=-1)
        
        # Feature 4: Boundaries
        if self.feature_config['use_word_boundary']:
            bp_emb = self.boundary_prev_emb(bound_prev)
            bn_emb = self.boundary_next_emb(bound_next)
            emb = torch.cat([emb, bp_emb, bn_emb], dim=-1)
        
        out1, _ = self.bilstm1(emb)
        out1 = self.dropout1(out1)
        out2, _ = self.bilstm2(out1)
        out2 = self.dropout2(out2)
        
        ff = self.relu(self.ff1(out2))
        ff = self.relu(self.ff2(ff))
        logits = self.out(ff)
        return logits


# ============================================================================
# Training Functions
# ============================================================================

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        x_ids, ngram_bi, ngram_tri, char_freq, bound_prev, bound_next, y = batch
        x_ids = x_ids.to(device)
        ngram_bi = ngram_bi.to(device)
        ngram_tri = ngram_tri.to(device)
        char_freq = char_freq.to(device)
        bound_prev = bound_prev.to(device)
        bound_next = bound_next.to(device)
        y = y.to(device)
        
        optimizer.zero_grad()
        logits = model(x_ids, ngram_bi, ngram_tri, char_freq, bound_prev, bound_next)
        
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(dataloader)

def eval_epoch(model, dataloader, criterion, device, pad_label_idx):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in dataloader:
            x_ids, ngram_bi, ngram_tri, char_freq, bound_prev, bound_next, y = batch
            x_ids = x_ids.to(device)
            ngram_bi = ngram_bi.to(device)
            ngram_tri = ngram_tri.to(device)
            char_freq = char_freq.to(device)
            bound_prev = bound_prev.to(device)
            bound_next = bound_next.to(device)
            y = y.to(device)
            
            logits = model(x_ids, ngram_bi, ngram_tri, char_freq, bound_prev, bound_next)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            total_loss += loss.item()
            
            preds = logits.argmax(dim=-1)
            mask = (y != pad_label_idx)
            correct += (preds[mask] == y[mask]).sum().item()
            total += mask.sum().item()
    
    acc = correct / total if total > 0 else 0.0
    return total_loss / len(dataloader), acc


# ============================================================================
# Run All Feature Experiments
# ============================================================================

print('\n' + '='*70)
print(' RUNNING FEATURE ABLATION STUDY')
print('='*70)

# Compute character frequencies once
char_freqs = compute_char_frequencies(train_tokens + val_tokens)

results = []

for exp_idx, feature_config in enumerate(FEATURE_EXPERIMENTS, 1):
    print(f'\n{"="*70}')
    print(f' EXPERIMENT {exp_idx}/{len(FEATURE_EXPERIMENTS)}: {feature_config["name"]}')
    print(f'{"="*70}')
    
    # Create datasets with current feature config
    train_dataset = FeatureDataset(train_tokens, train_labels, char_freqs, feature_config, MAXLEN)
    val_dataset = FeatureDataset(val_tokens, val_labels, char_freqs, feature_config, MAXLEN)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # Create model
    model = EnhancedBiLSTM(
        vocab_size=len(char2idx),
        num_labels=len(diac2idx),
        feature_config=feature_config
    ).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(ignore_index=diac2idx['<PAD_LABEL>'])
    
    best_val_acc = 0.0
    
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = eval_epoch(model, val_loader, criterion, device, diac2idx['<PAD_LABEL>'])
        
        print(f'Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}')
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            print(f'  New best accuracy: {best_val_acc:.4f}')
    
    # Store results
    results.append({
        'experiment': feature_config['name'],
        'best_accuracy': best_val_acc,
        'features': feature_config
    })
    
    print(f'\n Completed: {feature_config["name"]}')
    print(f'   Best Val Accuracy: {best_val_acc:.4f}')


# ============================================================================
# Results Summary
# ============================================================================

print('\n' + '='*70)
print('FEATURE ABLATION RESULTS SUMMARY')
print('='*70)

results_sorted = sorted(results, key=lambda x: x['best_accuracy'], reverse=True)

print('\n Ranked by Best Validation Accuracy:\n')
for i, result in enumerate(results_sorted, 1):
    print(f"{i}. {result['experiment']}")
    print(f"   Accuracy: {result['best_accuracy']:.4f}")
    features_used = [k.replace('use_', '') for k, v in result['features'].items() if v]
    print(f"   Features: {', '.join(features_used) if features_used else 'baseline only'}")
    print()

best_result = results_sorted[0]
print(f"Best Configuration: {best_result['experiment']}")
print(f"   Accuracy: {best_result['best_accuracy']:.4f}")